In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Data split

In [ ]:
import os
import shutil
import random

output_path = "/content/drive/MyDrive/Thesis_ANPR/dataset/split_anpr"
base_path="/content/drive/MyDrive/Thesis_ANPR/dataset/anpr_data"

for split in ["train", "val"]:
    os.makedirs(f"{output_path}/images/{split}", exist_ok=True)
    os.makedirs(f"{output_path}/labels/{split}", exist_ok=True)

images = [f for f in os.listdir(f"{base_path}/images") if f.endswith(".jpg")]
random.shuffle(images)

split_idx = int(0.8 * len(images))

train_files = images[:split_idx]
val_files = images[split_idx:]

def move(files, split):
    for f in files:
        img_src = f"{base_path}/images/{f}"
        lbl_src = f"{base_path}/labels/{f.replace('.jpg','.txt')}"

        img_dst = f"{output_path}/images/{split}/{f}"
        lbl_dst = f"{output_path}/labels/{split}/{f.replace('.jpg','.txt')}"

        if os.path.exists(lbl_src):
            shutil.copy(img_src, img_dst)
            shutil.copy(lbl_src, lbl_dst)

move(train_files, "train")
move(val_files, "val")

print("Dataset Ready ✅")

Dataset Ready ✅


In [ ]:
# generate yaml
import os

base_path = "/content/drive/MyDrive/Thesis_ANPR/dataset/split_anpr"

yaml_content = f"""
path: {base_path}
train: train/images
val: val/images

nc: 1
names: ['LP']
"""

yaml_file = os.path.join(base_path, "data.yaml")

with open(yaml_file, "w") as f:
    f.write(yaml_content.strip())

print("✅ data.yaml created successfully!")
!cat "{yaml_file}"

✅ data.yaml created successfully!
path: /content/drive/MyDrive/Thesis_ANPR/dataset/split_anpr
train: train/images
val: val/images

nc: 1
names: ['LP']

# EDA

In [ ]:
import os
import cv2

image_path = "/content/drive/MyDrive/Thesis_ANPR/dataset/anpr_data/images"

widths, heights = [], []

for img_name in os.listdir(image_path):
    img = cv2.imread(os.path.join(image_path, img_name))
    h, w, _ = img.shape
    widths.append(w)
    heights.append(h)

print("Total Images:", len(widths))
print("Avg Width:", sum(widths)//len(widths))
print("Avg Height:", sum(heights)//len(heights))

Total Images: 2215
Avg Width: 1146
Avg Height: 1284


# Model

In [2]:

!pip install ultralytics -q

from ultralytics import YOLO
import ultralytics

ultralytics.checks()

Ultralytics 8.4.39 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 43.0/112.6 GB disk)


In [3]:
!yolo checks
!yolo version

import os
from ultralytics import YOLO


data_yaml = "/content/drive/MyDrive/Thesis_ANPR/dataset/split_anpr/data.yaml"


print("YAML exists:", os.path.exists(data_yaml))
!cat "{data_yaml}"

Ultralytics 8.4.39 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 43.0/112.6 GB disk)

OS                     Linux-6.6.113+-x86_64-with-glibc2.35
Environment            Colab
Python                 3.12.13
Install                pip
Path                   /usr/local/lib/python3.12/dist-packages/ultralytics
RAM                    12.67 GB
Disk                   43.0/112.6 GB
CPU                    Intel Xeon CPU @ 2.00GHz
CPU count              2
GPU                    Tesla T4, 14913MiB
GPU count              1
CUDA                   12.8

numpy                  ✅ 2.0.2>=1.23.0
matplotlib             ✅ 3.10.0>=3.3.0
opencv-python          ✅ 4.13.0.92>=4.6.0
pillow                 ✅ 11.3.0>=7.1.2
pyyaml                 ✅ 6.0.3>=5.3.1
requests               ✅ 2.32.4>=2.23.0
scipy                  ✅ 1.16.3>=1.4.1
torch                  ✅ 2.10.0+cu128>=1.8.0
torch                  ✅ 2.10.0+cu128!=2.4.0,>=1.8.0; sys_platform == "win32

In [ ]:
base = "/content/drive/MyDrive/Thesis_ANPR/dataset/split_anpr"

print("=== Final Correct Structure Check ===")
print("Train images :", len(os.listdir(f"{base}/train/images")) if os.path.exists(f"{base}/train/images") else 0)
print("Train labels :", len(os.listdir(f"{base}/train/labels")) if os.path.exists(f"{base}/train/labels") else 0)
print("Val images   :", len(os.listdir(f"{base}/val/images")) if os.path.exists(f"{base}/val/images") else 0)
print("Val labels   :", len(os.listdir(f"{base}/val/labels")) if os.path.exists(f"{base}/val/labels") else 0)

=== Final Correct Structure Check ===
Train images : 1349
Train labels : 1349
Val images   : 340
Val labels   : 340


In [ ]:
base_path = "/content/drive/MyDrive/Thesis_ANPR/dataset/split_anpr"

yaml_content = f"""
path: {base_path}
train: train/images
val: val/images

nc: 1
names: ['LP']
"""

with open(f"{base_path}/data.yaml", "w") as f:
    f.write(yaml_content.strip())

print("✅ data.yaml updated!")
!cat "{base_path}/data.yaml"

✅ data.yaml updated!
path: /content/drive/MyDrive/Thesis_ANPR/dataset/split_anpr
train: train/images
val: val/images

nc: 1
names: ['LP']

In [4]:
from ultralytics import YOLO
import os
from google.colab import drive

drive.mount('/content/drive', force_remount=True)


data_yaml = "/content/drive/MyDrive/Thesis_ANPR/dataset/split_anpr/data.yaml"


print("✅ YAML exists:", os.path.exists(data_yaml))
!cat "{data_yaml}"

# মডেল
model = YOLO("yolov8n.pt")

# TRAINING
results = model.train(
    data=data_yaml,
    epochs=40,
    imgsz=640,
    batch=8,
    name="ANPR_YOLOv8n_baseline",
    patience=10,
    pretrained=True,
    optimizer="auto",
    seed=42,
    project="/content/drive/MyDrive/Thesis_ANPR/models",
    exist_ok=True,
    verbose=True
)

print("🎉 Training Completed Successfully!")

Mounted at /content/drive
✅ YAML exists: True
path: /content/drive/MyDrive/Thesis_ANPR/dataset/split_anpr
train: train/images
val: val/images

nc: 1
Ultralytics 8.4.39 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Thesis_ANPR/dataset/split_anpr/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, m

In [6]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# best.pt
source_path = "/content/drive/MyDrive/Thesis_ANPR/models/ANPR_YOLOv8n_baseline/weights/best.pt"

# Destination folder
dest_folder = "/content/drive/MyDrive/Thesis_ANPR/YOLOmodels"
os.makedirs(dest_folder, exist_ok=True)

# Best model
!cp "{source_path}" "{dest_folder}/yolov8n_baseline_best.pt"

print("✅ Baseline best model successfully saved to:")
print(f"{dest_folder}/yolov8n_baseline_best.pt")


if os.path.exists(f"{dest_folder}/yolov8n_baseline_best.pt"):
    print("✅ File exists and ready to use!")
    !ls -lh "{dest_folder}/yolov8n_baseline_best.pt"
else:
    print("❌ Something went wrong")

Mounted at /content/drive
✅ Baseline best model successfully saved to:
/content/drive/MyDrive/Thesis_ANPR/YOLOmodels/yolov8n_baseline_best.pt
✅ File exists and ready to use!
-rw------- 1 root root 6.0M Apr 19 17:56 /content/drive/MyDrive/Thesis_ANPR/YOLOmodels/yolov8n_baseline_best.pt


In [7]:
from ultralytics import YOLO
import os
import random
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

# Best model লোড
model = YOLO("/content/drive/MyDrive/Thesis_ANPR/YOLOmodels/yolov8n_baseline_best.pt")

# Validation images folder
val_img_folder = "/content/drive/MyDrive/Thesis_ANPR/dataset/split_anpr/val/images"

# Output folder যেখানে annotated images সেভ হবে
output_folder = "/content/drive/MyDrive/Thesis_ANPR/dataset/models/predictions_baseline"
os.makedirs(output_folder, exist_ok=True)

# Val folder থেকে random ৮-১০টা ইমেজ নাও
val_images = [f for f in os.listdir(val_img_folder) if f.endswith(('.jpg', '.jpeg', '.png'))]

if not val_images:
    print("❌ No images found in val/images folder!")
else:
    sample_images = random.sample(val_images, min(10, len(val_images)))

    print(f"✅ Running prediction on {len(sample_images)} images...\n")
    print(f"Annotated images will be saved in:\n{output_folder}\n")

    for img_name in sample_images:
        img_path = os.path.join(val_img_folder, img_name)
        print(f"Predicting → {img_name}")

        results = model.predict(
            img_path,
            save=True,                    # annotated image সেভ করবে
            conf=0.25,
            line_width=2,
            show_labels=True,
            show_conf=True,
            project=output_folder,        # Custom output folder
            name="",                      # ফোল্ডারের নাম খালি রাখা
            exist_ok=True
        )

    print("\n🎉 All predictions completed!")
    print(f"✅ Annotated images saved successfully in:")
    print(output_folder)

    # সেভ হওয়া ফাইলগুলো দেখি
    saved_files = os.listdir(output_folder)
    print(f"Total saved images: {len(saved_files)}")
    print("First 5 saved files:", saved_files[:5] if saved_files else "None")

Mounted at /content/drive
✅ Running prediction on 10 images...

Annotated images will be saved in:
/content/drive/MyDrive/Thesis_ANPR/dataset/models/predictions_baseline

Predicting → Vehicle1018_1952.jpg

image 1/1 /content/drive/MyDrive/Thesis_ANPR/dataset/split_anpr/val/images/Vehicle1018_1952.jpg: 640x320 1 LP, 39.0ms
Speed: 1.8ms preprocess, 39.0ms inference, 1.6ms postprocess per image at shape (1, 3, 640, 320)
Results saved to /content/drive/MyDrive/Thesis_ANPR/dataset/models/predictions_baseline/predict
Predicting → Vehicle725_3554.jpg

image 1/1 /content/drive/MyDrive/Thesis_ANPR/dataset/split_anpr/val/images/Vehicle725_3554.jpg: 640x480 1 LP, 40.5ms
Speed: 1.4ms preprocess, 40.5ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 480)
Results saved to /content/drive/MyDrive/Thesis_ANPR/dataset/models/predictions_baseline/predict
Predicting → Vehicle369_3158.jpg

image 1/1 /content/drive/MyDrive/Thesis_ANPR/dataset/split_anpr/val/images/Vehicle369_3158.jpg: 640x480 1

# vit now only for testing

In [ ]:
from ultralytics import YOLO
import os
import cv2
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

model = YOLO("/content/drive/MyDrive/Thesis_ANPR/models/yolov8n_baseline_best.pt")

# Input and Output folders
val_img_folder = "/content/drive/MyDrive/Thesis_ANPR/dataset/split_anpr/val/images"
crop_output_folder = "/content/drive/MyDrive/Thesis_ANPR/dataset/split_anpr/cropped_plates_test"

os.makedirs(crop_output_folder, exist_ok=True)

val_images = [f for f in os.listdir(val_img_folder) if f.endswith(('.jpg', '.jpeg', '.png'))]

print(f"Found {len(val_images)} images. Starting cropping...\n")

cropped_count = 0

for img_name in val_images[:15]:   # প্রথমে ১৫টা ইমেজ দিয়ে টেস্ট করি
    img_path = os.path.join(val_img_folder, img_name)
    results = model.predict(img_path, conf=0.25, verbose=False)

    for result in results:
        boxes = result.boxes
        if len(boxes) > 0:
            for box in boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                confidence = float(box.conf[0])

                # Original image load
                img = cv2.imread(img_path)
                cropped_plate = img[y1:y2, x1:x2]

                # Save cropped plate
                crop_name = f"{os.path.splitext(img_name)[0]}_crop_{cropped_count}.jpg"
                crop_path = os.path.join(crop_output_folder, crop_name)
                cv2.imwrite(crop_path, cropped_plate)

                cropped_count += 1
                print(f"✅ Cropped: {crop_name} | Conf: {confidence:.2f}")

print(f"\n🎉 Total {cropped_count} plates cropped and saved in:")
print(crop_output_folder)

Mounted at /content/drive
Found 340 images. Starting cropping...

✅ Cropped: Vehicle735_3565_crop_0.jpg | Conf: 0.78
✅ Cropped: Vehicle1362_2334_crop_1.jpg | Conf: 0.81
✅ Cropped: Vehicle532_3340_crop_2.jpg | Conf: 0.81
✅ Cropped: Vehicle1688_2695_crop_3.jpg | Conf: 0.81
✅ Cropped: Vehicle1688_2695_crop_4.jpg | Conf: 0.34
✅ Cropped: Vehicle387_3178_crop_5.jpg | Conf: 0.79
✅ Cropped: Vehicle1079_2019_crop_6.jpg | Conf: 0.78
✅ Cropped: Vehicle151_2498_crop_7.jpg | Conf: 0.77
✅ Cropped: Vehicle184_2864_crop_8.jpg | Conf: 0.81
✅ Cropped: Vehicle221_2995_crop_9.jpg | Conf: 0.74
✅ Cropped: Vehicle1060_1999_crop_10.jpg | Conf: 0.84
✅ Cropped: Vehicle1072_2012_crop_11.jpg | Conf: 0.75
✅ Cropped: Vehicle1214_2170_crop_12.jpg | Conf: 0.77
✅ Cropped: Vehicle138_2353_crop_13.jpg | Conf: 0.70
✅ Cropped: Vehicle650_3471_crop_14.jpg | Conf: 0.77
✅ Cropped: Vehicle1084_2025_crop_15.jpg | Conf: 0.78

🎉 Total 16 plates cropped and saved in:
/content/drive/MyDrive/Thesis_ANPR/dataset/split_anpr/cropped_p

In [ ]:
!pip install transformers torch torchvision -q

In [ ]:
from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer
from PIL import Image
import torch
import os

# Load pretrained model (ViT + Bangla capable)
processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")
tokenizer = AutoTokenizer.from_pretrained("sagorsarker/bangla-bert-base")   # Bangla BERT

# VisionEncoderDecoder model (testing purpose)
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")  # প্রথমে handwritten ট্রাই করি, পরে custom train করব

model.eval()

crop_folder = "/content/drive/MyDrive/Thesis_ANPR/dataset/split_anpr/cropped_plates_test"

for img_name in os.listdir(crop_folder)[:8]:   # প্রথম ৮টা cropped plate
    if not img_name.endswith('.jpg'):
        continue

    img_path = os.path.join(crop_folder, img_name)
    image = Image.open(img_path).convert("RGB")

    # Preprocess
    pixel_values = processor(images=image, return_tensors="pt").pixel_values

    # Generate text
    with torch.no_grad():
        generated_ids = model.generate(pixel_values, max_length=20)

    generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    print(f"Image: {img_name}")
    print(f"Predicted Text: {generated_text}")
    print("-" * 50)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/491 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ValueError: Input image size (224*224) doesn't match model (384*384).

In [ ]:
from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer
from PIL import Image
import torch
import os
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

# Processor & Tokenizer
processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-384")
tokenizer = AutoTokenizer.from_pretrained("sagorsarker/bangla-bert-base")

# Model load
model = VisionEncoderDecoderModel.from_encoder_decoder_pretrained(
    "google/vit-base-patch16-384",
    "sagorsarker/bangla-bert-base"
)

# === IMPORTANT: Generation Config Fix ===
model.generation_config.decoder_start_token_id = tokenizer.cls_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.sep_token_id
model.generation_config.max_length = 25
model.generation_config.early_stopping = True
model.generation_config.num_beams = 4
model.generation_config.length_penalty = 1.2
model.generation_config.no_repeat_ngram_size = 3

model.eval()

crop_folder = "/content/drive/MyDrive/Thesis_ANPR/dataset/split_anpr/cropped_plates_test"

print("✅ ViT + BanglaBERT Testing (Fixed Version) started...\n")

for img_name in sorted(os.listdir(crop_folder))[:10]:
    if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue

    img_path = os.path.join(crop_folder, img_name)
    image = Image.open(img_path).convert("RGB")

    pixel_values = processor(image, return_tensors="pt").pixel_values

    with torch.no_grad():
        generated_ids = model.generate(
            pixel_values,
            max_new_tokens=25,      # max_length এর পরিবর্তে এটা ব্যবহার করা ভালো
            num_beams=4,
            early_stopping=True
        )

    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    print(f"Image     : {img_name}")
    print(f"Predicted : {generated_text.strip()}")
    print("-" * 70)

Mounted at /content/drive


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-384
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertLMHeadModel LOAD REPORT from: sagorsarker/bangla-bert-base
Key                                                                | Status     | 
-------------------------------------------------------------------+------------+-
bert.pooler.dense.weight                                           | UNEXPECTED | 
cls.seq_relationship.weight                                        | UNEXPECTED | 
cls.seq_relationship.bias                                          | UNEXPECTED | 
bert.pooler.dense.bias                                             | UNEXPECTED | 
bert.encoder.layer.{0...11}.crossattention.self.key.weight         | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.output.dense.bias       | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.output.LayerNorm.bias   | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.self.value.bias         | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.output.LayerNorm.weight | MISSING    | 
bert.encoder.layer.{0...

✅ ViT + BanglaBERT Testing (Fixed Version) started...

Image     : Vehicle1060_1999_crop_10.jpg
Predicted : a a a b b b a a যদি যদি যদি,, তা তা তা হয় হয় হয় তবে তবে তবে
----------------------------------------------------------------------
Image     : Vehicle1072_2012_crop_11.jpg
Predicted : ,,... - ) ) ) ( ( (
----------------------------------------------------------------------
Image     : Vehicle1079_2019_crop_6.jpg
Predicted : ,,...??? - - - উপাত্ত উপাত্ত উপাত্ত - -
----------------------------------------------------------------------
Image     : Vehicle1084_2025_crop_15.jpg
Predicted : ,,... a a a,,, a a b b b a a
----------------------------------------------------------------------
Image     : Vehicle1214_2170_crop_12.jpg
Predicted : ...,,, a a a ) ) ) ( (
----------------------------------------------------------------------
Image     : Vehicle1362_2334_crop_1.jpg
Predicted : ,,,... ' ' ' - - - ' ',, ' '
----------------------------------------------------------------------